# Notebook 1: Experiment 1 — Same-Stock Prediction (80/20)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on each stock's daily data and predict its own future prices.  
**Train/Test Split:** 80/20 (chronological)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Stocks:** TLKM, BBCA, ASII, UNVR  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  


In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

set_seed()
set_ieee_style()
check_gpu()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.8
RATIO_LABEL = '80_20'
EXP_LABEL = f'Exp1_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 1 - Same Stock Prediction (80/20)")
print(f"Train ratio: {TRAIN_RATIO}, Test ratio: {1-TRAIN_RATIO}")


stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 60, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations

Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
Experiment 1 - Same Stock Prediction (80/20)
Train ratio: 0.8, Test ratio: 0.19999999999999996


In [2]:
# Load all daily data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)
print("\nAll daily data loaded!")


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

All daily data loaded!


## Run All Experiments

In [3]:
# ============================================================
# EXPERIMENT 1: Train and predict on same stock
# ============================================================
all_results = []
all_predictions = {}  # {stock: {model_type: (y_true, y_pred, dates)}}
all_histories = {}    # {stock: {model_type: history}}

for stock in STOCKS:
    print(f"\n############################################################")
    print(f"# STOCK: {stock}")
    print(f"############################################################")
    
    # Prepare data
    X_train, y_train, X_test, y_test, test_dates = prepare_same_stock_data(
        daily_data[stock], train_ratio=TRAIN_RATIO, lookback=LOOKBACK
    )
    print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")
    
    all_predictions[stock] = {}
    all_histories[stock] = {}
    
    for model_type in MODEL_TYPES:
        exp_name = f'{EXP_LABEL}_{stock}'
        
        y_true_inv, y_pred_inv, metrics, history = train_and_evaluate(
            model_type=model_type,
            X_train=X_train, y_train=y_train,
            X_test=X_test, y_test=y_test,
            experiment_name=exp_name,
            save_dir=f'models/{EXP_LABEL}',
            epochs=EPOCHS, batch_size=BATCH_SIZE
        )
        
        # Store results
        result = {'Stock': stock, 'Model': model_type, **metrics}
        all_results.append(result)
        all_predictions[stock][model_type] = (y_true_inv, y_pred_inv, test_dates)
        all_histories[stock][model_type] = history
        
        # Plot individual prediction
        plot_actual_vs_predicted(
            test_dates, y_true_inv, y_pred_inv,
            model_type, stock, EXP_LABEL,
            save_dir=f'figures/{EXP_LABEL}'
        )
        
        # Plot training history
        plot_training_history(
            history, model_type, stock, EXP_LABEL,
            save_dir=f'figures/{EXP_LABEL}'
        )

print("\n\nAll Experiment 1 (80/20) training complete!")



############################################################
# STOCK: TLKM
############################################################
  X_train: (4134, 60, 1), X_test: (1049, 60, 1)

Training BiLSTM for: Exp1_80_20_TLKM
  Train samples: 4134, Test samples: 1049
Epoch 1/100
59/59 [==============================] - ETA: 0s - loss: 0.0013
Epoch 1: val_loss improved from inf to 0.00024, saving model to models/Exp1_80_20\Exp1_80_20_TLKM_BiLSTM_best.keras
59/59 [==============================] - 11s 54ms/step - loss: 0.0013 - val_loss: 2.3713e-04
Epoch 2/100
59/59 [==============================] - ETA: 0s - loss: 1.4115e-04
Epoch 2: val_loss improved from 0.00024 to 0.00023, saving model to models/Exp1_80_20\Exp1_80_20_TLKM_BiLSTM_best.keras
59/59 [==============================] - 2s 32ms/step - loss: 1.4115e-04 - val_loss: 2.2652e-04
Epoch 3/100
59/59 [==============================] - ETA: 0s - loss: 1.3759e-04
Epoch 3: val_loss improved from 0.00023 to 0.00020, saving model to models

## Results Summary

In [4]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 1 - Same Stock Prediction (80/20)")

# Save results
results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")



  Experiment 1 - Same Stock Prediction (80/20)
Stock  Model        MSE     RMSE      MAE  MAPE (%)       R2  Training_Time_s  Epochs_Run
 TLKM BiLSTM  6507.3259  80.6680  61.7840    2.0469 0.961909            260.2         100
 TLKM  BiGRU  5784.6547  76.0569  57.8768    1.9099 0.966139            252.0         100
 TLKM   LSTM 11601.7542 107.7114  89.8112    2.9133 0.932089            139.0         100
 TLKM    GRU  8209.9453  90.6087  75.4171    2.4338 0.951943            137.6         100
 BBCA BiLSTM 56805.1783 238.3384 197.9622    2.3967 0.947444            257.3         100
 BBCA  BiGRU 41881.8134 204.6505 163.9348    2.0120 0.961251            251.0         100
 BBCA   LSTM 37351.9577 193.2665 148.0518    1.7607 0.965442            149.1         100
 BBCA    GRU 76330.2341 276.2793 225.3445    2.6717 0.929379            142.0         100
 ASII BiLSTM 13595.2684 116.5987  88.9684    1.8398 0.963202            206.6         100
 ASII  BiGRU  9678.1743  98.3777  73.7760    1.5390 

## Visualizations

In [5]:
# ============================================================
# ALL MODELS COMPARISON PER STOCK
# ============================================================
for stock in STOCKS:
    y_true = all_predictions[stock][MODEL_TYPES[0]][0]
    dates = all_predictions[stock][MODEL_TYPES[0]][2]
    preds = {mt: all_predictions[stock][mt][1] for mt in MODEL_TYPES}
    
    plot_all_models_comparison(
        dates, y_true, preds, stock, EXP_LABEL,
        save_dir=f'figures/{EXP_LABEL}'
    )

print("All comparison plots saved!")


  Figure saved: figures/Exp1_80_20/Exp1_80_20_TLKM_all_models.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_BBCA_all_models.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_ASII_all_models.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_UNVR_all_models.png
All comparison plots saved!


In [6]:
# ============================================================
# METRICS BAR CHARTS
# ============================================================
for metric in ['MSE', 'RMSE', 'MAE', 'MAPE (%)', 'R2']:
    plot_metrics_comparison_bar(
        results_df, metric, EXP_LABEL,
        group_col='Stock', save_dir=f'figures/{EXP_LABEL}'
    )

print("All metrics bar charts saved!")


  Figure saved: figures/Exp1_80_20/Exp1_80_20_MSE_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_RMSE_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_MAE_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_MAPE_pct_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_R2_comparison.png
All metrics bar charts saved!


In [7]:
# ============================================================
# SUMMARY: BEST MODEL PER STOCK
# ============================================================
print("\n" + "="*60)
print("  BEST MODEL PER STOCK (by RMSE)")
print("="*60)
for stock in STOCKS:
    stock_results = results_df[results_df['Stock'] == stock]
    best_idx = stock_results['RMSE'].idxmin()
    best = stock_results.loc[best_idx]
    print(f"  {stock}: {best['Model']} (RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")

print("\n  BEST MODEL PER STOCK (by R² Score)")
print("="*60)
for stock in STOCKS:
    stock_results = results_df[results_df['Stock'] == stock]
    best_idx = stock_results['R2'].idxmax()
    best = stock_results.loc[best_idx]
    print(f"  {stock}: {best['Model']} (R²={best['R2']:.6f}, RMSE={best['RMSE']:.4f})")



  BEST MODEL PER STOCK (by RMSE)
  TLKM: BiGRU (RMSE=76.0569, R²=0.966139)
  BBCA: LSTM (RMSE=193.2665, R²=0.965442)
  ASII: GRU (RMSE=85.6514, R²=0.980143)
  UNVR: GRU (RMSE=90.3589, R²=0.990614)

  BEST MODEL PER STOCK (by R² Score)
  TLKM: BiGRU (R²=0.966139, RMSE=76.0569)
  BBCA: LSTM (R²=0.965442, RMSE=193.2665)
  ASII: GRU (R²=0.980143, RMSE=85.6514)
  UNVR: GRU (R²=0.990614, RMSE=90.3589)


## Interactive Prediction Visualization (Plotly)
Zoom in, pan, and explore the price predictions with hover details.

In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# INTERACTIVE VISUALIZATION - ACTUAL VS PREDICTED (Plotly)
# ============================================================

for stock in STOCKS:
    print(f"\nGenerating interactive plot for {stock}...")
    
    y_true = all_predictions[stock][MODEL_TYPES[0]][0]
    dates = all_predictions[stock][MODEL_TYPES[0]][2]
    
    # Create subplots (one for each model)
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=MODEL_TYPES,
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]],
        vertical_spacing=0.12,
        horizontal_spacing=0.1
    )
    
    row_col_pairs = [(1, 1), (1, 2), (2, 1), (2, 2)]
    
    for idx, model_type in enumerate(MODEL_TYPES):
        row, col = row_col_pairs[idx]
        y_pred = all_predictions[stock][model_type][1]
        
        # Actual prices
        fig.add_trace(
            go.Scatter(
                x=dates, y=y_true,
                name='Actual Price',
                mode='lines',
                line=dict(color='blue', width=2),
                hovertemplate='<b>Actual Price</b><br>Date: %{x}<br>Price: %{y:.2f}<extra></extra>',
            ),
            row=row, col=col
        )
        
        # Predicted prices
        fig.add_trace(
            go.Scatter(
                x=dates, y=y_pred,
                name=f'{model_type} Prediction',
                mode='lines',
                line=dict(color='red', width=2, dash='dash'),
                hovertemplate='<b>Predicted Price</b><br>Date: %{x}<br>Price: %{y:.2f}<extra></extra>',
            ),
            row=row, col=col
        )
        
        # Update axes labels
        fig.update_xaxes(title_text="Date", row=row, col=col)
        fig.update_yaxes(title_text="Price", row=row, col=col)
    
    # Update layout
    fig.update_layout(
        title=f'<b>{stock} - Actual vs Predicted Prices (Experiment 1 - 80/20)</b>',
        height=900,
        template='plotly_white',
        hovermode='x unified',
        showlegend=True,
        legend=dict(
            orientation="v",
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.02,
        ),
        font=dict(size=10)
    )
    
    # Save as HTML
    fig.write_html(f'figures/{EXP_LABEL}/interactive_{stock}_all_models.html')
    fig.show()

print("\n✓ All interactive plots generated and saved!")
print(f"  Location: figures/{EXP_LABEL}/interactive_*.html")


Generating interactive plot for TLKM...



Generating interactive plot for BBCA...



Generating interactive plot for ASII...



Generating interactive plot for UNVR...



✓ All interactive plots generated and saved!
  Location: figures/Exp1_80_20/interactive_*.html


In [10]:
# ============================================================
# INTERACTIVE VISUALIZATION - TOGGLE MODELS WITH BUTTONS
# ============================================================

for stock in STOCKS:
    print(f"\nGenerating toggle model plot for {stock}...")
    
    y_true = all_predictions[stock][MODEL_TYPES[0]][0]
    dates = all_predictions[stock][MODEL_TYPES[0]][2]
    
    fig = go.Figure()
    
    # Add actual price (always visible)
    fig.add_trace(
        go.Scatter(
            x=dates, y=y_true,
            name='Actual Price',
            mode='lines',
            line=dict(color='blue', width=2.5),
            hovertemplate='<b>Actual Price</b><br>Date: %{x|%Y-%m-%d}<br>Price: IDR %{y:,.2f}<extra></extra>',
            visible=True
        )
    )
    
    # Add predictions for each model (togglable)
    for model_type in MODEL_TYPES:
        y_pred = all_predictions[stock][model_type][1]
        
        fig.add_trace(
            go.Scatter(
                x=dates, y=y_pred,
                name=f'{model_type} Prediction',
                mode='lines',
                line=dict(width=2),
                hovertemplate=f'<b>{model_type}</b><br>Date: %{{x|%Y-%m-%d}}<br>Price: IDR %{{y:,.2f}}<extra></extra>',
                visible=True
            )
        )
    
    # Create buttons for model selection
    buttons = [
        dict(
            label="All Models",
            method="update",
            args=[{"visible": [True] * (len(MODEL_TYPES) + 1)},
                  {"title": f"<b>{stock} - All Models vs Actual Price</b>"}]
        )
    ]
    
    for i, model_type in enumerate(MODEL_TYPES):
        visible = [True] + [False] * len(MODEL_TYPES)
        visible[i + 1] = True
        buttons.append(
            dict(
                label=model_type,
                method="update",
                args=[{"visible": visible},
                      {"title": f"<b>{stock} - {model_type} vs Actual Price</b>"}]
            )
        )
    
    # Update layout with buttons
    fig.update_layout(
        updatemenus=[
            dict(
                type="dropdown",
                direction="down",
                x=0.01,
                y=0.99,
                showactive=True,
                buttons=buttons,
                bgcolor="lightgray",
                bordercolor="gray",
                borderwidth=1,
            )
        ],
        title=f"<b>{stock} - All Models vs Actual Price</b>",
        xaxis_title="Date",
        yaxis_title="Price (IDR)",
        template="plotly_white",
        hovermode="x unified",
        height=600,
        font=dict(size=11),
        xaxis=dict(
            rangeslider=dict(visible=False),
            type="date"
        ),
        yaxis=dict(
            gridwidth=1,
            gridcolor="lightgray"
        )
    )
    
    # Save as HTML
    fig.write_html(f'figures/{EXP_LABEL}/interactive_{stock}_toggle.html')
    fig.show()

print("\n✓ All toggle model plots generated and saved!")
print(f"  Location: figures/{EXP_LABEL}/interactive_*_toggle.html")


Generating toggle model plot for TLKM...



Generating toggle model plot for BBCA...



Generating toggle model plot for ASII...



Generating toggle model plot for UNVR...



✓ All toggle model plots generated and saved!
  Location: figures/Exp1_80_20/interactive_*_toggle.html
